In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

Concatenação dos datasets

In [ ]:
import pandas as pd
from pathlib import Path

PROCESSED = Path("/content/drive/MyDrive/TCC_Chatbot/datasets/processed")
MERGED = Path("/content/drive/MyDrive/TCC_Chatbot/datasets/merged")

MERGED.mkdir(parents=True, exist_ok=True)

arquivos = [
    "codefeedback_python.parquet",
    "glaive_python.parquet",
    "stackoverflow_python.parquet",
    "faq_python.parquet",
    "dataset_python_qa.parquet",
    "code_qa_updated.parquet"
]

colunas = ["question", "answer", "code", "source"]

dfs = [
    pd.read_parquet(PROCESSED / arq)[colunas]
    for arq in arquivos
]

banco = pd.concat(dfs, ignore_index=True)

print(f"Total de registros: {len(banco)}")
print("\nColunas:")
print(banco.columns)

print("\nInformações do DataFrame:")
print(banco.info())

print("\nValores nulos:")
print(banco.isnull().sum())

print("\nAmostra:")
print(banco.sample(5))

arquivo_saida = MERGED / "banco_concatenado.parquet"

banco.to_parquet(
    arquivo_saida,
    index=False
)

print(f"\nBanco concatenado salvo em:\n{arquivo_saida}")

Verificação da estrutura do dataset concatenado

In [ ]:
import pandas as pd

files = [
    "/content/drive/MyDrive/TCC_Chatbot/datasets/merged/banco_concatenado.parquet"
]

for file in files:
    print("=" * 100)
    print(f"Arquivo: {file}")

    df = pd.read_parquet(file)

    print("\nColunas:")
    print(df.columns.tolist())

    print("\nTipos:")
    print(df.dtypes)

    print("\nPrimeiras linhas:")
    print(df.head())

    print("\nQuantidade de registros:")
    print(len(df))

    print("\nValores nulos:")
    print(df.isnull().sum())

    print("\nInformações do DataFrame:")
    print(df.info())

Remoção das duplicatas do dataset concatenado

In [ ]:
import pandas as pd
from pathlib import Path

MERGED = Path("/content/drive/MyDrive/TCC_Chatbot/datasets/merged")

banco = pd.read_parquet(MERGED / "banco_concatenado.parquet")

antes = len(banco)

banco = (
    banco
    .drop_duplicates(subset=["question", "answer"], keep="first")
    .reset_index(drop=True)
)

depois = len(banco)

print(f"Registros antes: {antes:,}")
print(f"Registros depois: {depois:,}")
print(f"Duplicatas removidas: {antes - depois:,}")

print("\nColunas:")
print(banco.columns)

print("\nValores nulos:")
print(banco.isnull().sum())

print("\nAmostra:")
print(banco.sample(5))

arquivo_saida = MERGED / "banco_concatenado_sem_duplicatas.parquet"

banco.to_parquet(
    arquivo_saida,
    index=False
)

print(f"\nBanco salvo em:\n{arquivo_saida}")

Verificação dos assuntos do dataset concatenado

In [ ]:
import pandas as pd
from pathlib import Path

MERGED = Path("/content/drive/MyDrive/TCC_Chatbot/datasets/merged")

banco = pd.read_parquet(
    MERGED / "banco_concatenado_sem_duplicatas.parquet"
)

quantidade = 30

amostra = banco.sample(n=quantidade, random_state=42)

print(f"Exibindo {quantidade} registros aleatórios do banco.\n")

for i, (_, linha) in enumerate(amostra.iterrows(), start=1):

    print("=" * 120)
    print(f"REGISTRO {i}")

    print("\nPERGUNTA:")
    print(linha["question"])

    print("\nRESPOSTA:")
    print(linha["answer"])

    print("\nCÓDIGO:")
    if str(linha["code"]).strip():
        print(linha["code"])
    else:
        print("(Sem código associado)")

    print("\nFONTE:")
    print(linha["source"])

    print("\n")

    print("Quantidade de registros por fonte:\n")

    print(banco["source"].value_counts())

    print("Perguntas vazias:",
      (banco["question"].str.strip() == "").sum())

    print("Respostas vazias:",
      (banco["answer"].str.strip() == "").sum())

    print("Códigos vazios:",
      (banco["code"].str.strip() == "").sum())

Filtragem de linguagem python

In [ ]:
import pandas as pd
import re

df = pd.read_parquet(
    "/content/drive/MyDrive/TCC_Chatbot/datasets/merged/banco_concatenado_sem_duplicatas.parquet"
)

padroes_outros = {

    "javascript": [
        r"\bnode\.?js\b",
        r"\bexpress\.js\b",
        r"require\(",
        r"console\.log",
        r"\bfunction\s+\w+\(",
        r"\bconst\s+\w+\s*=",
        r"\blet\s+\w+\s*=",
        r"=>"
    ],

    "java": [
        r"\bpublic\s+class\b",
        r"System\.out\.println",
        r"\bjava\.util\b",
        r"\bArrayList\b",
        r"\bHashMap\b"
    ],

    "php": [
        r"<\?php",
        r"\$\w+\s*=",
        r"echo\s+"
    ],

    "csharp": [
        r"Console\.WriteLine",
        r"\busing\s+System\b",
        r"\bnamespace\s+\w+"
    ],

    "cpp": [
        r"#include\s*<iostream>",
        r"\bstd::\w+",
        r"\bcout\s*<<"
    ]
}

texto_resposta = (
    df["answer"]
    .fillna("")
    .str.lower()
)


remover = pd.Series(
    False,
    index=df.index
)

glaive_mask = (
    df["source"] == "glaive_python"
)


relatorio = {}


for linguagem, lista in padroes_outros.items():

    regex = "|".join(lista)

    encontrados = (
        texto_resposta.str.contains(
            regex,
            regex=True,
            na=False
        )
        &
        glaive_mask
    )

    quantidade = encontrados.sum()

    relatorio[linguagem] = quantidade

    print(
        linguagem,
        quantidade
    )

    remover |= encontrados


print("\nRegistros removidos:")
print(relatorio)

df_removidos = df[remover]

df_final = df[~remover]


print("\nAntes:", len(df))
print("Removidos:", len(df_removidos))
print("Depois:", len(df_final))

df_removidos.to_parquet(
    "/content/drive/MyDrive/TCC_Chatbot/datasets/merged/registros_removidos_outros_idiomas.parquet",
    index=False
)

df_final.to_parquet(
    "/content/drive/MyDrive/TCC_Chatbot/datasets/merged/banco_python_final.parquet",
    index=False
)

Validação e filtragem adicional sobre linguagens suspeitas encontradas no dataset

In [ ]:
import pandas as pd
from pathlib import Path


MERGED = Path(
    "/content/drive/MyDrive/TCC_Chatbot/datasets/merged"
)


banco = pd.read_parquet(
    MERGED / "banco_python_final.parquet"
)


codeqa = banco[
    banco["source"] == "code_qa_updated"
].copy()


texto = (
    codeqa["answer"].fillna("")
    + " "
    + codeqa["code"].fillna("")
).str.lower()


padroes = {

    "cpp": [
        r"#include\s*<[^>]+>",
        r"std::",
        r"\bcout\s*<<",
        r"\bcin\s*>>"
    ],

    "java": [
        r"public\s+static\s+void\s+main\s*\(",
        r"system\.out\.println",
        r"import\s+java\.",
        r"package\s+\w+"
    ],

    "csharp": [
        r"console\.writeline\s*\(",
        r"using\s+system;",
        r"namespace\s+\w+"
    ],

    "php": [
        r"<\?php",
        r"\$\w+\s*="
    ],

    "javascript": [
        r"console\.log\s*\(",
        r"\bvar\s+\w+\s*=",
        r"\blet\s+\w+\s*="
    ],

    "ruby": [
        r"\bputs\s+",
        r"\bdef\s+\w+"
    ]
}


suspeitos = []


for linguagem, lista in padroes.items():

    regex = "|".join(lista)

    encontrados = codeqa[
        texto.str.contains(
            regex,
            regex=True,
            na=False
        )
    ].copy()

    encontrados["linguagem_suspeita"] = linguagem

    print(
        linguagem,
        len(encontrados)
    )

    suspeitos.append(encontrados)


suspeitos = pd.concat(
    suspeitos,
    ignore_index=True
)


print(
    "\nTotal suspeitos:",
    len(suspeitos)
)


suspeitos[[
    "question",
    "answer",
    "code",
    "source",
    "linguagem_suspeita"
]].head(20)

Remoção dessas linguagens suspeitas do dataset concatenado

In [ ]:
import pandas as pd
from pathlib import Path


MERGED = Path(
    "/content/drive/MyDrive/TCC_Chatbot/datasets/merged"
)


banco = pd.read_parquet(
    MERGED / "banco_python_final.parquet"
)


banco["indice_original"] = banco.index


codeqa = banco[
    banco["source"] == "code_qa_updated"
].copy()


texto_codigo = (
    codeqa["code"]
    .fillna("")
    .str.lower()
)


padroes = {

    "cpp": [
        r"#include\s*<[^>]+>",
        r"\bstd::",
        r"\bcout\s*<<",
        r"\bcin\s*>>",
        r"\busing\s+namespace\s+std\b"
    ],

    "java": [
        r"public\s+static\s+void\s+main\s*\(",
        r"system\.out\.println",
        r"import\s+java\.",
        r"package\s+\w+"
    ],

    "csharp": [
        r"console\.writeline\s*\(",
        r"using\s+system;",
        r"namespace\s+\w+"
    ],

    "php": [
        r"<\?php",
        r"\$\w+\s*="
    ],

    "javascript": [
        r"console\.log\s*\(",
        r"\bvar\s+\w+\s*=",
        r"\blet\s+\w+\s*="
    ]
}


suspeitos_lista = []


for linguagem, lista in padroes.items():

    regex = "|".join(lista)

    encontrados = codeqa[
        texto_codigo.str.contains(
            regex,
            regex=True,
            na=False
        )
    ].copy()


    encontrados["linguagem_suspeita"] = linguagem


    print(
        linguagem,
        len(encontrados)
    )


    suspeitos_lista.append(encontrados)



suspeitos = pd.concat(
    suspeitos_lista,
    ignore_index=True
)


print(
    "\nTotal suspeitos:",
    len(suspeitos)
)


suspeitos.to_parquet(
    MERGED / "registros_removidos_outras_linguagens.parquet",
    index=False
)


indices_remover = suspeitos["indice_original"].unique()


banco_python_final = banco[
    ~banco["indice_original"].isin(indices_remover)
]


print("\nAntes:", len(banco))
print("Removidos:", len(indices_remover))
print("Depois:", len(banco_python_final))


banco_python_final = banco_python_final.drop(
    columns=["indice_original"]
)


banco_python_final.to_parquet(
    MERGED / "banco_python_final.parquet",
    index=False
)


print("\nBanco final salvo!")

Verificação da estrutura e quantidade de dados de cada dataset usado na concatenação

In [ ]:
df = pd.read_parquet(
    MERGED / "banco_python_final.parquet"
)

print(df["source"].value_counts())
print(df.columns)

Verificação das respostas do dataset concatenado

In [ ]:
import pandas as pd
from pathlib import Path
import re

MERGED = Path(
    "/content/drive/MyDrive/TCC_Chatbot/datasets/merged"
)

banco = pd.read_parquet(
    MERGED / "banco_python_final.parquet"
)

texto = banco["answer"].fillna("").str.lower()

topicos = {
    "Variáveis": [
        "variable", "variables", "assignment"
    ],
    "Tipos de dados": [
        "int", "float", "str", "string", "bool", "boolean"
    ],
    "Entrada e saída": [
        "input", "print"
    ],
    "Condicionais": [
        "if", "elif", "else"
    ],
    "Laços": [
        "for", "while", "break", "continue"
    ],
    "Funções": [
        "def", "function", "return", "parameter", "argument"
    ],
    "Listas": [
        "list", "append", "extend", "insert"
    ],
    "Tuplas": [
        "tuple"
    ],
    "Dicionários": [
        "dict", "dictionary", "keys", "values"
    ],
    "Conjuntos": [
        "set"
    ],
    "Strings": [
        "split", "replace", "strip", "startswith", "endswith"
    ],
    "Arquivos": [
        "open", "read", "write", "file"
    ],
    "Exceções": [
        "try", "except", "finally", "exception"
    ],
    "Classes": [
        "class", "__init__", "object"
    ],
    "Módulos": [
        "import", "module", "package"
    ]
}

print(f"Total de respostas: {len(banco):,}\n")

for assunto, palavras in topicos.items():

    padrao = r"\b(" + "|".join(map(re.escape, palavras)) + r")\b"

    quantidade = texto.str.contains(
        padrao,
        regex=True,
        na=False
    ).sum()

    percentual = quantidade / len(banco) * 100

    print(
        f"{assunto:20} "
        f"{quantidade:>8,} respostas "
        f"({percentual:.2f}%)"
    )